# 07 - CNN-BiLSTM character branch (tier A)

**GPU runtime required** - the notebook refuses to train on CPU. Runtime ->
Change runtime type -> GPU, then Run all. **Resumes**: each (split, seed) run
saves predictions and weights; completed runs are skipped, and an interrupted
run resumes from its last epoch checkpoint on Drive.

Why a learned character model: the handcrafted lexical features still lose
0.04 ROC from random to family-disjoint even without `tld` - length,
consonant runs and n-gram scores partly fingerprint generators. The question
this notebook answers is whether a representation learned from raw characters
generalises to unseen families *better* than handcrafted statistics. It is
evaluated family-disjoint first; the random-split number is context.

Focal loss rather than SMOTE: interpolating character embeddings would
synthesise domains that cannot exist. Data is staged to local SSD before
training - reading batches off the Drive mount would starve the GPU.

In [ ]:
# --- standard header ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'
if os.path.isdir(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)
else:
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone','-q',f'https://{TOKEN}@{URL}',REPO], check=True)
sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
!pip -q install pyarrow zstandard

In [ ]:
import torch, numpy as np, pandas as pd, time, json
from pathlib import Path
from torch.utils.data import TensorDataset, DataLoader
from src.models.cnn_bilstm import CharEncoder, CNNBiLSTM, FocalLoss
from src.evaluate import splits, metrics, predictions
from src.utils import manifest as mf, io as uio

assert torch.cuda.is_available(), (
    'No GPU. Runtime -> Change runtime type -> GPU, then Run all. '
    'Training 1.2M domains x 3 seeds x 2 splits on CPU is not feasible.')
DEV = 'cuda'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

local = uio.stage_local(f"{P['data']['features']}/fused_v1.parquet", P['local']['data'])
df = pd.read_parquet(local, columns=['domain','label'])
print('staged locally:', local, df.shape)

## Configuration

Read from `configs/cnn_bilstm.yaml`; epochs capped for wall-clock. With
~2,400 batches per epoch on the full training part, one epoch is roughly
30-60 seconds on a T4; 25 epochs x 3 seeds x 2 splits fits an afternoon.

In [ ]:
import yaml
cfg = yaml.safe_load(open(f'{REPO}/configs/cnn_bilstm.yaml'))
cfg['train']['epochs'] = 25
cfg['train']['early_stopping_patience'] = 5
print(json.dumps(cfg, indent=1))

enc = CharEncoder(cfg['input']['charset'], cfg['input']['max_length'])

# pass only what the constructor accepts (the model is always bidirectional)
import inspect
MODEL_KW = {k: v for k, v in cfg['model'].items()
            if k in inspect.signature(CNNBiLSTM.__init__).parameters}
print('model kwargs:', MODEL_KW)
CKPT_DIR = Path(P['artifacts']['checkpoints']); CKPT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR = Path(P['artifacts']['predictions']); MODEL_DIR = Path(P['artifacts']['models'])

In [ ]:
def encode_frame(frame):
    X = torch.from_numpy(enc.encode_batch(frame['domain'].values))
    y = torch.tensor(frame['label'].values, dtype=torch.float32)
    return X, y

def loader(X, y, shuffle, bs):
    return DataLoader(TensorDataset(X, y), batch_size=bs, shuffle=shuffle,
                      pin_memory=True, num_workers=2)

@torch.no_grad()
def predict(model, dl):
    model.eval(); out = []
    for xb, _ in dl:
        out.append(torch.sigmoid(model(xb.to(DEV, non_blocking=True))).float().cpu().numpy())
    return np.concatenate(out)

## Train grid (resumable at epoch granularity)

A per-run checkpoint on Drive holds model, optimiser, scheduler, epoch, best
score and patience counter. If the runtime dies mid-run, the run continues
from the last completed epoch on the next Run all.

In [ ]:
SPLITS = ['family_disjoint_v1', 'random_v1']      # honest split first
SEEDS  = [42, 43, 44]
bs, epochs = cfg['train']['batch_size'], cfg['train']['epochs']

for split_name in SPLITS:
    sp = splits.load_split(P['data']['splits'], split_name); d = sp['domains']
    tr = df[df['domain'].isin(d['train'])]; va = df[df['domain'].isin(d['val'])]
    te = df[df['domain'].isin(d['test'])]
    Xtr, ytr = encode_frame(tr); Xva, yva = encode_frame(va); Xte, yte = encode_frame(te)
    dl_va, dl_te = loader(Xva, yva, False, 4096), loader(Xte, yte, False, 4096)

    for seed in SEEDS:
        run_id = f'cnnbilstm_tierA_{split_name}_s{seed}'
        if (PRED_DIR/f'{run_id}.parquet').exists():
            print('SKIP (done)', run_id); continue

        seeds.set_all(seed)
        dl_tr = loader(Xtr, ytr, True, bs)
        model = CNNBiLSTM(enc.vocab_size, **MODEL_KW).to(DEV)
        opt = torch.optim.AdamW(model.parameters(), lr=cfg['train']['lr'],
                                weight_decay=cfg['train']['weight_decay'])
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        crit = FocalLoss(gamma=cfg['train']['focal_gamma'])
        scaler = torch.amp.GradScaler('cuda')

        ckpt_path = CKPT_DIR/f'{run_id}.pt'
        state = {'epoch': 0, 'best': -1.0, 'patience': 0}
        if ckpt_path.exists():
            ck = torch.load(ckpt_path, map_location=DEV, weights_only=False)
            model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
            sched.load_state_dict(ck['sched']); state = ck['state']
            print(f'RESUME {run_id} from epoch {state["epoch"]} (best {state["best"]:.4f})')
        best_path = CKPT_DIR/f'{run_id}_best.pt'

        for epoch in range(state['epoch'], epochs):
            model.train(); t0 = time.time(); tot = 0.0
            for xb, yb in dl_tr:
                xb, yb = xb.to(DEV, non_blocking=True), yb.to(DEV, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.autocast('cuda', dtype=torch.float16):
                    loss = crit(model(xb), yb)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                tot += loss.item()
            sched.step()
            m_va = metrics.evaluate(yva.numpy().astype(int), predict(model, dl_va))
            improved = m_va['roc_auc'] > state['best']
            if improved:
                state['best'], state['patience'] = m_va['roc_auc'], 0
                torch.save(model.state_dict(), best_path)
            else:
                state['patience'] += 1
            state['epoch'] = epoch + 1
            torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                        'sched': sched.state_dict(), 'state': state}, ckpt_path)
            print(f'{run_id} ep{epoch+1:02d} loss={tot/len(dl_tr):.4f} '
                  f'val_roc={m_va["roc_auc"]:.4f} val_fpr95={m_va["fpr_at_95_tpr"]:.4f} '
                  f'{"*" if improved else ""} {time.time()-t0:.0f}s')
            if state['patience'] >= cfg['train']['early_stopping_patience']:
                print('early stop'); break

        model.load_state_dict(torch.load(best_path, map_location=DEV))
        scores = predict(model, dl_te)
        m = metrics.evaluate(yte.numpy().astype(int), scores); m['best_val_roc'] = state['best']
        m['epochs_trained'] = state['epoch']
        predictions.save(run_id, PRED_DIR, te['domain'].values, yte.numpy().astype(int), scores)
        torch.save(model.state_dict(), MODEL_DIR/f'{run_id}.pt')
        mf.record(P['manifest'], run_id, 'cnnbilstm_tierA', cfg, split_name,
                  sp['split_file'], m, seed, repo_root=REPO)
        ckpt_path.unlink(missing_ok=True)
        print(f'DONE {run_id:44s} roc={m["roc_auc"]:.4f} fpr@95={m["fpr_at_95_tpr"]:.4f}')
print('grid complete')

## Character model vs handcrafted lexical (same tier, same splits)

In [ ]:
man = mf.load_manifest(P['manifest'])
fams = ['xgb_tierA_lexical','xgb_tierA_notld','cnnbilstm_tierA']
a = man[man['run_family'].isin(fams)]
tab = (a.groupby(['run_family','split_name'])
        [['metrics.roc_auc','metrics.fpr_at_95_tpr']].agg(['mean','std']).round(4))
display(tab)

roc = a.groupby(['run_family','split_name'])['metrics.roc_auc'].mean().unstack('split_name')
roc['generalisation_drop'] = (roc['random_v1'] - roc['family_disjoint_v1']).round(4)
display(roc.round(4))
tab.to_csv(Path(P['results']['tables'])/'table_tierA_char_vs_lexical.csv')

---

**How to read it.** The column that matters is `family_disjoint_v1`, and the
row comparison is CNN-BiLSTM vs `xgb_tierA_notld`. If the learned
representation closes the generalisation gap, the character branch is the
right DGA-regime component for the fused score. If it does not, handcrafted
lexical without tld is - and that is reported as such.

**Next:** `08_fusion` - regime-aware fusion over the probe universe.